# QuantBench 全模型训练回测台账 v1

目标：把 QuantBench 注册表中的所有模型一次性训练、预测、回测，并生成逐模型运行台账。这份 notebook 是本次单次研究证据，业务功能保留在 `src/model`、`src/backtest`、`src/analysis` 和 `src/objective` 中。

## 实验契约

- Baseline：`run_quantbench_matrix_smoke` 当前主线矩阵。
- Role：diagnose，证明每个注册模型都进入同一训练、回测、analysis、objective、promotion 记录链路。
- Target window：2026-01-01 到 2026-06-09，训练截止 2026-04-30。
- Changed layer：model。
- Acceptance：`model_run_ledger_summary.json` 中 `all_models_completed=True`，且 missing_models 为空。
- Failure mode：有模型只出现在 registry，却没有预测、回测、objective/promotion 或 manifest 记录。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.model import QUANTBENCH_MODEL_NAMES
from src.model.matrix import run_quantbench_matrix_smoke

CARD_DIR = repo_root / 'research' / 'quantbench_all_model_run_ledger_v1'
START = '2026-01-01'
END = '2026-06-09'
TRAIN_END = '2026-04-30'
len(QUANTBENCH_MODEL_NAMES), QUANTBENCH_MODEL_NAMES[:5], QUANTBENCH_MODEL_NAMES[-5:]

(32,
 ('logistic_clf',
  'ridge_ic',
  'ridge_mse',
  'random_forest_clf',
  'hist_gradient_mse'),
 ('rsr_mse', 'hypergraph_mse', 'thgnn_mse', 'sthan_mse', 'sthgcn_mse'))

In [2]:
outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start=START,
    end=END,
    train_end=TRAIN_END,
    model_names=QUANTBENCH_MODEL_NAMES,
)

ledger = outputs.model_run_ledger.copy()
ledger_summary = json.loads((CARD_DIR / 'model_run_ledger_summary.json').read_text(encoding='utf-8'))
analysis_summary = json.loads((CARD_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
ledger_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'model.run_ledger_summary.v1',
 'expected_model_count': 32,
 'ledger_model_count': 32,
 'registered_model_count': 32,
 'trained_model_count': 32,
 'predicted_model_count': 32,
 'backtested_model_count': 32,
 'objective_evaluated_model_count': 32,
 'promotion_evaluated_model_count': 32,
 'analysis_recorded_model_count': 32,
 'manifested_model_count': 32,
 'proxy_model_count': 0,
 'native_model_count': 32,
 'paper_fidelity_counts': {'native_graph': 9,
  'native_library': 10,
  'native_sequence': 7,
  'native_torch': 6},
 'adapter_statuses': ['paper_strict_adapter'],
 'missing_models': [],
 'all_models_completed': True}

In [3]:
review_columns = [
    'model', 'base_model', 'family', 'objective', 'paper_fidelity', 'adapter_status',
    'trained', 'predicted', 'backtested', 'objective_evaluated',
    'promotion_evaluated', 'analysis_recorded', 'manifested', 'comparison_rank',
    'return_pct', 'max_drawdown_pct', 'sharpe', 'stability_score', 'promotion_decision',
]
ledger.loc[:, review_columns].sort_values('comparison_rank').head(12)

,model,base_model,family,objective,paper_fidelity,adapter_status,trained,predicted,backtested,objective_evaluated,promotion_evaluated,analysis_recorded,manifested,comparison_rank,return_pct,max_drawdown_pct,sharpe,stability_score,promotion_decision
18,mlp_mixer_mse,mlp_mixer_mse,sequence,mse,native_sequence,paper_strict_adapter,True,True,True,True,True,True,True,1,55.285791,20.475703,4.121763,60.725394,promote_candidate
27,rsr_mse,rsr_mse,graph,mse,native_graph,paper_strict_adapter,True,True,True,True,True,True,True,2,45.636395,22.981182,3.683452,43.672385,promote_candidate
17,tcn_mse,tcn_mse,sequence,mse,native_torch,paper_strict_adapter,True,True,True,True,True,True,True,3,44.126973,24.842772,3.516625,41.208206,promote_candidate
9,mlp_mse,mlp_mse,neural_tabular,mse,native_library,paper_strict_adapter,True,True,True,True,True,True,True,4,45.180246,24.485225,3.757595,40.141611,promote_candidate
22,fedformer_mse,fedformer_mse,sequence,mse,native_sequence,paper_strict_adapter,True,True,True,True,True,True,True,5,19.532160,7.586366,3.648972,36.581539,promote_candidate
15,darnn_mse,darnn_mse,sequence,mse,native_torch,paper_strict_adapter,True,True,True,True,True,True,True,6,37.592797,26.828884,3.161380,34.094988,promote_candidate
29,thgnn_mse,thgnn_mse,graph,mse,native_graph,paper_strict_adapter,True,True,True,True,True,True,True,7,44.019103,28.171913,3.439516,33.689408,promote_candidate
16,hawkes_gru_mse,hawkes_gru_mse,sequence,mse,native_torch,paper_strict_adapter,True,True,True,True,True,True,True,8,31.407994,14.707499,3.249848,32.568884,promote_candidate
6,lightgbm_mse,lightgbm_mse,boosting,mse,native_library,paper_strict_adapter,True,True,True,True,True,True,True,9,36.732852,26.563401,3.197491,31.645040,promote_candidate
4,hist_gradient_mse,hist_gradient_mse,tree,mse,native_library,paper_strict_adapter,True,True,True,True,True,True,True,10,34.421338,26.563956,3.076637,27.687527,promote_candidate


In [4]:
comparison = pd.read_csv(CARD_DIR / 'model_comparison.csv')
regime_metrics = pd.read_csv(CARD_DIR / 'regime_metrics.csv')
window_metrics = pd.read_csv(CARD_DIR / 'window_metrics.csv')
objective_scores = pd.read_csv(CARD_DIR / 'objective_scores.csv')
promotion_scores = pd.read_csv(CARD_DIR / 'promotion_scores.csv')

top_models = comparison.sort_values('rank').head(8)
regime_counts = regime_metrics.groupby('regime')['model'].nunique().sort_index().to_dict()
window_counts = window_metrics.groupby('window')['model'].nunique().sort_index().to_dict()
objective_counts = objective_scores['objective'].value_counts().sort_index().to_dict()
promotion_counts = promotion_scores['promotion_decision'].value_counts().sort_index().to_dict()
{
    'top_models': top_models[['rank', 'model', 'return_pct', 'max_drawdown_pct', 'sharpe', 'stability_score']].to_dict(orient='records'),
    'regime_counts': regime_counts,
    'window_counts': window_counts,
    'objective_counts': objective_counts,
    'promotion_counts': promotion_counts,
}

{'top_models': [{'rank': 1,
   'model': 'mlp_mixer_mse',
   'return_pct': 55.2857907793225,
   'max_drawdown_pct': 20.47570313055833,
   'sharpe': 4.121762824528974,
   'stability_score': 60.7253938909699},
  {'rank': 2,
   'model': 'rsr_mse',
   'return_pct': 45.63639496969558,
   'max_drawdown_pct': 22.98118156962913,
   'sharpe': 3.6834522076394474,
   'stability_score': 43.67238548690951},
  {'rank': 3,
   'model': 'tcn_mse',
   'return_pct': 44.12697276767366,
   'max_drawdown_pct': 24.842772004446047,
   'sharpe': 3.5166252743776147,
   'stability_score': 41.208205976788406},
  {'rank': 4,
   'model': 'mlp_mse',
   'return_pct': 45.18024593040848,
   'max_drawdown_pct': 24.48522533343227,
   'sharpe': 3.757594955504647,
   'stability_score': 40.141611296734226},
  {'rank': 5,
   'model': 'fedformer_mse',
   'return_pct': 19.532160077246985,
   'max_drawdown_pct': 7.586365629337033,
   'sharpe': 3.648972143690933,
   'stability_score': 36.581538818957576},
  {'rank': 6,
   'model'

In [5]:
from IPython.display import Markdown

required_flags = [
    'trained', 'predicted', 'backtested', 'objective_evaluated',
    'promotion_evaluated', 'analysis_recorded', 'manifested',
]
incomplete = ledger.loc[~ledger[required_flags].astype(bool).all(axis=1), 'model'].tolist()
top_row = comparison.sort_values('rank').iloc[0]
native_count = int(ledger_summary['native_model_count'])
proxy_count = int(ledger_summary['proxy_model_count'])
summary_payload = {
    'schema_version': 'research.quantbench_all_model_run_ledger_v1',
    'card_dir': str(CARD_DIR),
    'start': START,
    'end': END,
    'train_end': TRAIN_END,
    'expected_model_count': ledger_summary['expected_model_count'],
    'trained_model_count': ledger_summary['trained_model_count'],
    'backtested_model_count': ledger_summary['backtested_model_count'],
    'all_models_completed': ledger_summary['all_models_completed'],
    'missing_models': ledger_summary['missing_models'],
    'native_model_count': native_count,
    'proxy_model_count': proxy_count,
    'top_model': str(top_row['model']),
    'top_return_pct': float(top_row['return_pct']),
    'top_max_drawdown_pct': float(top_row['max_drawdown_pct']),
    'top_sharpe': float(top_row['sharpe']),
    'top_stability_score': float(top_row['stability_score']),
    'artifact_links': {
        'model_run_ledger': 'model_run_ledger.csv',
        'model_run_ledger_summary': 'model_run_ledger_summary.json',
        'model_comparison': 'model_comparison.csv',
        'regime_metrics': 'regime_metrics.csv',
        'window_metrics': 'window_metrics.csv',
        'objective_scores': 'objective_scores.csv',
        'promotion_scores': 'promotion_scores.csv',
        'analysis_manifest': 'analysis_manifest.json',
        'artifact_manifest': 'artifact_manifest.json',
    },
}
(CARD_DIR / 'notebook_summary.json').write_text(
    json.dumps(summary_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

conclusion = f'''## 中文结论

- 做了什么：本 notebook 调用 `src.model.matrix.run_quantbench_matrix_smoke`，对 QuantBench 注册表中的 {ledger_summary['expected_model_count']} 个模型完成训练、预测、回测、analysis、objective 和 promotion 记录。
- 核心指标：trained={ledger_summary['trained_model_count']}，backtested={ledger_summary['backtested_model_count']}，all_models_completed={ledger_summary['all_models_completed']}，missing_models={ledger_summary['missing_models']}。
- 模型实现状态：native={native_count}，proxy={proxy_count}；当前 QuantBench 注册模型已无 `proxy_feature_adapter`，后续重点转为长窗稳健性、成本滑点和 production paper/sim 验证。
- 当前窗口最强模型：{top_row['model']}，return_pct={float(top_row['return_pct']):.4f}，max_drawdown_pct={float(top_row['max_drawdown_pct']):.4f}，sharpe={float(top_row['sharpe']):.4f}，stability_score={float(top_row['stability_score']):.4f}。
- Regime/区间差异：`regime_metrics.csv`、`window_metrics.csv`、`window_regime_metrics.csv` 已按同一 analysis contract 输出，后续 dashboard 和人工 review 应读取 manifest/ledger，不直接猜测散落文件。
- 是否晋升：本次是 coverage/diagnostic 运行，不直接晋升生产模型；下一步应优先做长窗 walk-forward、压力窗口和 dashboard/promotion 读取验证。
- 风险：训练窗口较短，SOXL 2026 年样本偏强；ledger 证明链路完整，不证明模型具备长期泛化能力。
'''
Markdown(conclusion)

## 中文结论

- 做了什么：本 notebook 调用 `src.model.matrix.run_quantbench_matrix_smoke`，对 QuantBench 注册表中的 32 个模型完成训练、预测、回测、analysis、objective 和 promotion 记录。
- 核心指标：trained=32，backtested=32，all_models_completed=True，missing_models=[]。
- 模型实现状态：native=32，proxy=0；当前 QuantBench 注册模型已无 `proxy_feature_adapter`，后续重点转为长窗稳健性、成本滑点和 production paper/sim 验证。
- 当前窗口最强模型：mlp_mixer_mse，return_pct=55.2858，max_drawdown_pct=20.4757，sharpe=4.1218，stability_score=60.7254。
- Regime/区间差异：`regime_metrics.csv`、`window_metrics.csv`、`window_regime_metrics.csv` 已按同一 analysis contract 输出，后续 dashboard 和人工 review 应读取 manifest/ledger，不直接猜测散落文件。
- 是否晋升：本次是 coverage/diagnostic 运行，不直接晋升生产模型；下一步应优先做长窗 walk-forward、压力窗口和 dashboard/promotion 读取验证。
- 风险：训练窗口较短，SOXL 2026 年样本偏强；ledger 证明链路完整，不证明模型具备长期泛化能力。


## 中文结论

- 做了什么：本 notebook 调用 `src.model.matrix.run_quantbench_matrix_smoke`，对 QuantBench 注册表中的全部 32 个模型完成训练、预测、回测、analysis、objective 和 promotion 记录。
- 核心指标：`model_run_ledger_summary.json` 显示 expected=32、trained=32、backtested=32、all_models_completed=true、missing_models=[]。
- 模型实现状态：native=32、proxy=0；`ridge_ic` 已升级为 `RidgeICRegressor`，`extra_trees_rank` 已升级为 `ExtraTreesRankRegressor`，sequence/graph 系列也已进入 native_sequence/native_graph/native_torch 路径，当前 QuantBench 注册模型已无 `proxy_feature_adapter`。
- Regime/区间差异：`regime_metrics.csv`、`window_metrics.csv`、`window_regime_metrics.csv`、`model_effect_deltas.csv` 已按统一 analysis contract 输出；dashboard 和人工 review 应读取 manifest/ledger。
- 是否晋升：否。本次是 coverage/diagnostic 运行，用来证明链路完整，不直接晋升任何生产模型。
- 风险：训练窗口较短，SOXL 2026 年样本偏强；ledger 证明模型链路完整，不证明长期泛化能力。
- 下一步：把候选模型重新纳入更长 walk-forward、成本滑点、压力窗口和 paper/sim replay，并让 dashboard/promotion 只读取 manifest/ledger。